# Recommendation Systems for Movies

Two MovieLens datasets are used here. (link : https://www.kaggle.com/rounakbanik/the-movies-dataset/data)



In [1]:
#pip install surprise

In [2]:
#importing required libraries 
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.options.display.max_columns = None

from scipy import stats
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import wordnet
from surprise import Reader, Dataset, SVD #(if this code doesn't run, please remove the "#" before pip install surprise in the previous cell and run the cell)
from surprise.model_selection import cross_validate


import warnings; warnings.simplefilter('ignore')

## Content Based Recommendation model

![](https://johnolamendy.files.wordpress.com/2015/10/01.png)

** Limitation of Popularity model ** <br>
It gives the same recommendation to everyone, regardless of the user's personal interest. <br>

_For example:_ If a person who loves romantic movies were to look at Top 15 romantic movies, and he wouldn't probably like most of the listed movies. So, he were to go one step further and look at movie lists by genre, he wouldn't still be getting the interesting recommendations. 

Therefore, let's build a model that computes similarity between movies based on certain metrics and suggests movies that are most similar to a particular movie that a user liked. For that we have to consider metadata (or content), hence, it also known as **Content Based Filtering.**

Two Content Based Recommendation is implemented based on different contents:
1. Description Based (content: Movie Overviews and Taglines)
2. Meta Data Based (content : Movie Cast, Crew, Keywords and Genre)

** Note **: A small movie data set is used due to limiting computing power available to me. 

In [3]:
#Please enter the file directory name where you have stored the dataset (make sure you store the jupyter notebook in the same folders with datasets)
small_mdf = pd.read_csv('links_small.csv')
small_mdf = small_mdf[small_mdf['tmdbId'].notnull()]['tmdbId'].astype('int')

Before extracting small data set, we need to make sure that the ID column of our main dataframe is clean and of type integer. To do this, let us try to perform an integer conversion of our IDs and if an exception is raised,we will replace the ID with NaN. We will then proceed to drop these rows from our dataframe.

In [4]:
def convert_int(x):
    try:
        return int(x)
    except:
        return np.nan

In [5]:
#Please enter the file directory name where you have stored the dataset (make sure you store the jupyter notebook in the same folders with datasets)
m_df = pd.read_csv('movies_metadata.csv')
m_df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [6]:
m_df['id'] = m_df['id'].apply(convert_int)

In [7]:
m_df[m_df['id'].isnull()]

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
19730,- Written by Ørnås,0.065736,/ff9qCepilowshEtG2GYWwzt2bs4.jpg,"[{'name': 'Carousel Productions', 'id': 11176}...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",NaN,0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Midnight Man,False,6.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29503,Rune Balot goes to a casino connected to the ...,1.931659,/zV8bHuSL6WXoD6FWogP9j4x80bL.jpg,"[{'name': 'Aniplex', 'id': 2883}, {'name': 'Go...","[{'iso_3166_1': 'US', 'name': 'United States o...",NaN,0,68.0,"[{'iso_639_1': 'ja', 'name': '日本語'}]",Released,NaN,Mardock Scramble: The Third Exhaust,False,7.0,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35587,Avalanche Sharks tells the story of a bikini ...,2.185485,/zaSf5OG7V8X8gqFvly88zDdRm46.jpg,"[{'name': 'Odyssey Media', 'id': 17161}, {'nam...","[{'iso_3166_1': 'CA', 'name': 'Canada'}]",NaN,0,82.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Beware Of Frost Bites,Avalanche Sharks,False,4.3,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
m_df = m_df.drop([19730, 29503, 35587])
m_df['id'] = m_df['id'].astype('int')

In [9]:
sm_df = m_df[m_df['id'].isin(small_mdf)]
sm_df.shape

(9099, 24)

**9099** movies avaiable in our small movies metadata dataset 

### 1. Description Based Recommendation

In [10]:
sm_df['tagline'] = sm_df['tagline'].fillna('')
sm_df['description'] = sm_df['overview'] + sm_df['tagline']
sm_df['description'] = sm_df['description'].fillna('')

#### Compute TF-IDF matrix 

In [1]:
tf = TfidfVectorizer(analyzer='word',ngram_range=(1, 2),min_df=1, stop_words='english')
tfidf_matrix = tf.fit_transform(sm_df['description'])
tfidf_matrix.shape

NameError: name 'TfidfVectorizer' is not defined

#### Cosine Similarity

The Cosine Similarity is used to calculate a numeric quantity that denotes the similarity between two movies. Mathematically, it is defined as follows:

$cosine(x,y) = \frac{x. y^\intercal}{||x||.||y||} $

Since the TF-IDF Vectorizer is used, calculating the Dot Product will directly give us the Cosine Similarity Score. Therefore, sklearn's **linear_kernel** is used instead of cosine_similarities as it's much faster.

In [12]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [13]:
cosine_sim[0]

array([1.        , 0.00680476, 0.        , ..., 0.        , 0.00344913,
       0.        ])

We now have a pairwise cosine similarity matrix for all the movies in our dataset. The next step is to write a function that returns the 50 most similar movies based on the cosine similarity score.

In [14]:
sm_df = sm_df.reset_index()
titles = sm_df['title']
indices = pd.Series(sm_df.index, index=sm_df['title'])

In [15]:
def get_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:51]
    movie_indices = [i[0] for i in sim_scores]
    return titles.iloc[movie_indices]

Let us now try and get the top 10 recommendations for a few movies.

In [16]:
get_recommendations('The Godfather').head(10)

973      The Godfather: Part II
8387                 The Family
3509                       Made
4196         Johnny Dangerously
29               Shanghai Triad
5667                       Fury
2412             American Movie
1582    The Godfather: Part III
4221                    8 Women
2159              Summer of Sam
Name: title, dtype: object

In [17]:
get_recommendations('The Dark Knight').head(10)

7931                      The Dark Knight Rises
132                              Batman Forever
1113                             Batman Returns
8227    Batman: The Dark Knight Returns, Part 2
7565                 Batman: Under the Red Hood
524                                      Batman
7901                           Batman: Year One
2579               Batman: Mask of the Phantasm
2696                                        JFK
8165    Batman: The Dark Knight Returns, Part 1
Name: title, dtype: object

We see that for **The Dark Knight**, our system is able to identify it as a Batman film and subsequently recommend other Batman films as its top recommendations. But unfortunately, that is all this system can do at the moment. This is not of much use to most people as it doesn't take into considerations very important features such as cast, crew, director and genre, which determine the rating and the popularity of a movie. Someone who liked **The Dark Knight** probably likes it more because of Nolan and would hate **Batman Forever** and every other substandard movie in the Batman Franchise.

Therefore, we are going to use much more suggestive metadata than **Overview** and **Tagline**. In the next subsection, Let's build a more sophisticated recommender that takes **genre**, **keywords**, **cast** and **crew** into consideration.

## Collaborative Filtering

![](https://cdn-images-1.medium.com/max/1600/1*6_NlX6CJYhtxzRM-t6ywkQ.png)



** Limitation of content based recommendation**: 
- It is only capable of suggesting movies which are *close* to a certain movie. That is, it is not capable of capturing interest and providing recommendations across genres.

- It doesn't capture the personal intrest and biases of a user. Anyone querying on model for recommendations based on a movie will receive the same recommendations for that movie, regardless of who he is.

Therefore, in this section, we will use a technique called **Collaborative Filtering** to make recommendations to Movie Watchers. Collaborative Filtering is based on the idea that users similar to a me can be used to predict how much I will like a particular product or service those users have used/experienced but I have not.

We will use the **Surprise** library that used extremely powerful algorithms like **Singular Value Decomposition (SVD)** to minimise RMSE (Root Mean Square Error) and give great recommendations.

In [18]:
reader = Reader()


In [19]:
#Please enter the file directory name where you have stored the dataset (make sure you store the jupyter notebook in the same folders with datasets)
ratings = pd.read_csv('ratings_small.csv')
ratings.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [20]:
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)


In [21]:
svd = SVD()
cross_validate(svd, data, measures=['RMSE', 'MAE'])

{'test_rmse': array([0.89558922, 0.89561264, 0.89240077, 0.89923733, 0.89736575]),
 'test_mae': array([0.68890837, 0.68820226, 0.68688337, 0.69405136, 0.69109416]),
 'fit_time': (3.459554672241211,
  3.5093464851379395,
  3.5231103897094727,
  3.52089524269104,
  3.6027443408966064),
 'test_time': (0.10711932182312012,
  0.1074838638305664,
  0.10662603378295898,
  0.10601449012756348,
  0.10784602165222168)}

We get a mean **Root Mean Sqaure Error** of 0.8965 which is more than good enough right now. Let us now train on dataset and arrive at predictions.

Let us pick user 5000 and check the ratings he has given.

In [22]:
ratings[ratings['userId'] == 1]

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205
5,1,1263,2.0,1260759151
6,1,1287,2.0,1260759187
7,1,1293,2.0,1260759148
8,1,1339,3.5,1260759125
9,1,1343,2.0,1260759131


In [23]:
svd.predict(1, 302, 3)

Prediction(uid=1, iid=302, r_ui=3, est=2.9731108667527244, details={'was_impossible': False})

For movie with ID 302, we get an estimated prediction of **2.74**. One startling feature of this recommender system is that it doesn't care what the movie is (or what it contains). It works purely on the basis of an assigned movie ID and tries to predict ratings based on how the other users have predicted the movie.

## Hybrid Recommender


we build a simple hybrid recommender that brings together techniques we have implemented in the content based and collaborative filter based engines. This is how it will work:

* **Input:** User ID and the Title of a Movie
* **Output:** Similar movies sorted on the basis of expected ratings by that particular user.

In [24]:
def convert_int(x):
    try:
        return int(x)
    except:
        return np.nan

In [25]:
#Please enter the file directory name where you have stored the dataset (make sure you store the jupyter notebook in the same folders with datasets)
id_map = pd.read_csv('links_small.csv')[['movieId', 'tmdbId']]
id_map['tmdbId'] = id_map['tmdbId'].apply(convert_int)
id_map.columns = ['movieId', 'id']
id_map = id_map.merge(sm_df[['title', 'id']], on='id').set_index('title')
#id_map = id_map.set_index('tmdbId')

In [26]:
indices_map = id_map.set_index('id')

In [27]:
def hybrid(userId, title):
    idx = indices[title]
    tmdbId = id_map.loc[title]['id']
    #print(idx)
    movie_id = id_map.loc[title]['movieId']
    
    sim_scores = list(enumerate(cosine_sim[int(idx)]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:26]
    movie_indices = [i[0] for i in sim_scores]
    
    movies = sm_df.iloc[movie_indices][['title', 'vote_count', 'vote_average', 'id']]
    movies['est'] = movies['id'].apply(lambda x: svd.predict(userId, indices_map.loc[x]['movieId']).est)
    movies = movies.sort_values('est', ascending=False)
    return movies.head(10)

In [28]:
hybrid(1, 'Avatar')

,title,vote_count,vote_average,id,est
2059,The Matrix,9079.0,7.9,603,3.438109
975,A Grand Day Out,199.0,7.4,530,3.278060
6105,A Trip to the Moon,314.0,7.9,775,3.236226
1898,A Simple Plan,191.0,6.9,10223,3.131021
4643,"Heaven Knows, Mr. Allison",27.0,6.8,37103,3.023330
3360,The Dish,62.0,6.6,5257,2.970860
4804,Avalon,93.0,6.8,10881,2.961690
2910,Pandora and the Flying Dutchman,19.0,6.5,38688,2.951401
4328,Dog Soldiers,227.0,6.7,11880,2.810458
2854,The Hidden,85.0,6.7,12476,2.808704


In [29]:
hybrid(500, 'Avatar')

,title,vote_count,vote_average,id,est
975,A Grand Day Out,199.0,7.4,530,3.299192
4643,"Heaven Knows, Mr. Allison",27.0,6.8,37103,3.294488
5044,The Men,18.0,6.5,1882,3.237413
5229,Ambush,13.0,6.3,49320,3.138732
3015,House Party 2,22.0,4.7,16096,3.118488
1898,A Simple Plan,191.0,6.9,10223,3.087477
4328,Dog Soldiers,227.0,6.7,11880,3.070781
7050,Pride and Glory,243.0,6.3,13150,3.043841
3360,The Dish,62.0,6.6,5257,3.034897
7763,Hanna,1284.0,6.5,50456,3.021158


We see that for our hybrid recommender, we get different recommendations for different users although the movie is the same. Hence, our recommendations are more personalized and tailored towards particular users.

## Conclusion

In this notebook, we have built 4 different recommendation engines based on different ideas and algorithms. They are as follows:

1. **Content Based Recommender:** We built two content based engines; one that took movie overview and taglines as input and the other which took metadata such as cast, crew, genre and keywords to come up with predictions. We also deviced a simple filter to give greater preference to movies with more votes and higher ratings.
2. **Collaborative Filtering:** We used the powerful Surprise Library to build a collaborative filter based on single value decomposition. The RMSE obtained was less than 1 and the engine gave estimated ratings for a given user and movie.
3. **Hybrid Engine:** We brought together ideas from content and collaborative filterting to build an engine that gave movie suggestions to a particular user based on the estimated ratings that it had internally calculated for that user.

# Assignment



**Task 1**

Build a *Content based recommendation system* using the following dataset in kaggle : 
https://www.kaggle.com/stefanoleone992/imdb-extensive-dataset



In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the Dataset
# We only need specific columns to build the content profile.
# Note: The full dataset is quite large. For performance on a standard machine, 
# you might want to filter for movies with a minimum number of votes or from a specific decade.
file_path = 'IMDb movies.csv'
cols = ['title', 'genre', 'director', 'actors', 'description', 'votes']

df = pd.read_csv(file_path, usecols=cols)

# Optional: Filter to the top 20,000 most voted movies to prevent memory limits 
# when creating the n x n similarity matrix.
df = df[df['votes'] > 10000].sort_values('votes', ascending=False).head(20000)
df = df.reset_index(drop=True)

# 2. Data Preprocessing
# Fill any missing values with an empty string so it doesn't break our text processing
features = ['genre', 'director', 'actors', 'description']
for feature in features:
    df[feature] = df[feature].fillna('')

# Create a combined feature string (the "soup")
def create_soup(x):
    return f"{x['genre']} {x['director']} {x['actors']} {x['description']}"

df['soup'] = df.apply(create_soup, axis=1)

# 3. Text Vectorization
# Initialize TF-IDF Vectorizer. 
# stop_words='english' removes common words like 'the', 'and', 'is'
tfidf = TfidfVectorizer(stop_words='english')

# Construct the required TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(df['soup'])

# 4. Compute Cosine Similarity
# Using linear_kernel is equivalent to cosine_similarity for TF-IDF vectors 
# but is slightly faster, though cosine_similarity is perfectly fine here too.
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a reverse mapping of movie titles to dataframe indices
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

# 5. Build the Recommendation Function
def get_recommendations(title, cosine_sim=cosine_sim, df=df, indices=indices):
    # Check if the movie exists in our dataset
    if title not in indices:
        return f"Movie '{title}' not found in the dataset."
    
    # Get the index of the movie that matches the title
    idx = indices[title]

    # Get the pairwise similarity scores of all movies with that movie
    # Enumerate adds the index to the score: (index, score)
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar movies (ignoring the first one, which is the movie itself)
    sim_scores = sim_scores[1:11]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 10 most similar movies
    return df['title'].iloc[movie_indices]

# --- Testing the System ---
print("Recommendations for 'The Dark Knight':")
print(get_recommendations('The Dark Knight'))

Recommendations for 'The Dark Knight':
Movie 'The Dark Knight' not found in the dataset.


**Task 2**

Build a *Collaborative recommendation system* using the following dataset in kaggle : 
https://www.kaggle.com/stefanoleone992/imdb-extensive-dataset

In [7]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------
movies = pd.read_csv("IMDb movies.csv", low_memory=False)
ratings = pd.read_csv("IMDb ratings.csv", low_memory=False)

# --------------------------------------------------
# 2. Merge the movie metadata with ratings
# --------------------------------------------------
df = movies.merge(
    ratings,
    how="inner",
    left_on="imdb_title_id",
    right_on="imdb_title_id"
)

# Keep only useful columns
keep_cols = [
    "imdb_title_id",
    "title",
    "year",
    "genre",
    "weighted_average_vote",
    "total_votes",
    "males_allages_avg_vote",
    "males_18age_avg_vote",
    "males_30age_avg_vote",
    "males_45age_avg_vote",
    "females_allages_avg_vote",
    "females_18age_avg_vote",
    "females_30age_avg_vote",
    "females_45age_avg_vote",
    "top1000_voters_rating",
    "us_voters_rating",
    "non_us_voters_rating"
]

df = df[[c for c in keep_cols if c in df.columns]].copy()

# --------------------------------------------------
# 3. Basic cleaning
# --------------------------------------------------
# Convert numeric columns safely
numeric_cols = [c for c in df.columns if c not in ["imdb_title_id", "title", "genre", "year"]]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Use only movies with enough votes for stability
if "total_votes" in df.columns:
    df = df[df["total_votes"] >= 1000].copy()

# Remove rows without a title
df = df.dropna(subset=["title"])

# --------------------------------------------------
# 4. Build pseudo-user ratings table
# --------------------------------------------------
pseudo_user_cols = [
    c for c in [
        "males_allages_avg_vote",
        "males_18age_avg_vote",
        "males_30age_avg_vote",
        "males_45age_avg_vote",
        "females_allages_avg_vote",
        "females_18age_avg_vote",
        "females_30age_avg_vote",
        "females_45age_avg_vote",
        "top1000_voters_rating",
        "us_voters_rating",
        "non_us_voters_rating"
    ] if c in df.columns
]

# rows = pseudo-users, cols = movies
ratings_matrix = df.set_index("title")[pseudo_user_cols].T

# Fill missing ratings with each pseudo-user's mean
ratings_matrix = ratings_matrix.apply(
    lambda row: row.fillna(row.mean()),
    axis=1
)

# Optional fallback if an entire row is null
ratings_matrix = ratings_matrix.fillna(ratings_matrix.mean(axis=1), axis=0).fillna(0)

# --------------------------------------------------
# 5. Matrix factorization (collaborative-style)
# --------------------------------------------------
n_components = min(8, min(ratings_matrix.shape) - 1) if min(ratings_matrix.shape) > 1 else 1
svd = TruncatedSVD(n_components=n_components, random_state=42)

latent = svd.fit_transform(ratings_matrix)          # pseudo-user latent features
movie_factors = svd.components_.T                  # movie latent features

movie_titles = ratings_matrix.columns.tolist()
movie_factors_df = pd.DataFrame(movie_factors, index=movie_titles)

# --------------------------------------------------
# 6. Movie-to-movie similarity
# --------------------------------------------------
movie_similarity = cosine_similarity(movie_factors_df)
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_titles,
    columns=movie_titles
)

def recommend_similar_movies(movie_title, top_n=10):
    if movie_title not in movie_similarity_df.index:
        return pd.DataFrame({"message": [f"Movie '{movie_title}' not found."]})

    sims = movie_similarity_df[movie_title].sort_values(ascending=False)
    sims = sims.drop(index=movie_title, errors="ignore").head(top_n)

    results = df[df["title"].isin(sims.index)][["title", "year", "genre", "weighted_average_vote", "total_votes"]].drop_duplicates("title")
    results["similarity"] = results["title"].map(sims)
    results = results.sort_values(by="similarity", ascending=False)

    return results.reset_index(drop=True)

# --------------------------------------------------
# 7. Segment-specific recommendation
# --------------------------------------------------
def recommend_for_segment(segment_name, top_n=10, min_votes=5000):
  if segment_name not in pseudo_user_cols:
    return pd.DataFrame({"message": [f"Segment '{segment_name}' not found."]})

  # Build scores using unique movie IDs, not titles
  segment_scores = df[
    ["imdb_title_id", "title", "year", "genre", "weighted_average_vote", "total_votes", segment_name]].copy()
  segment_scores = segment_scores.dropna(subset=[segment_name])

  if "total_votes" in segment_scores.columns:
    segment_scores = segment_scores[segment_scores["total_votes"] >= min_votes]

  # Sort by the selected segment's rating
  segment_scores = segment_scores.sort_values(
    by=[segment_name, "weighted_average_vote", "total_votes"],
    ascending=[False, False, False]
  )

  # Rename the chosen segment column to segment_score
  segment_scores = segment_scores.rename(columns={segment_name: "segment_score"})

  return segment_scores[
    ["imdb_title_id", "title", "year", "genre", "weighted_average_vote", "total_votes", "segment_score"]
  ].head(top_n).reset_index(drop=True)

# --------------------------------------------------
# 8. Example usage
# --------------------------------------------------
print("Recommendations similar to The Dark Knight:")
print(recommend_similar_movies("The Dark Knight", top_n=10))

print("\nTop recommendations for top1000_voters_rating:")
print(recommend_for_segment("top1000_voters_rating", top_n=10))

Recommendations similar to The Dark Knight:
                              message
0  Movie 'The Dark Knight' not found.

Top recommendations for top1000_voters_rating:
  imdb_title_id                            title  year  \
0     tt0068646                       Il padrino  1972   
1     tt0111161             Le ali della libertà  1994   
2     tt0071562            Il padrino - Parte II  1974   
3     tt0108052                 Schindler's List  1993   
4     tt0050083             La parola ai giurati  1957   
5     tt0060196  Il buono, il brutto, il cattivo  1966   
6     tt0102926      Il silenzio degli innocenti  1991   
7     tt0082971    I predatori dell'arca perduta  1981   
8     tt0078748                            Alien  1979   
9     tt0468569              Il cavaliere oscuro  2008   

                       genre  weighted_average_vote  total_votes  \
0               Crime, Drama                    9.2      1572674   
1                      Drama                    9.3      